<a href="https://colab.research.google.com/github/ruchit0002/Autoreq-agentic-data-integrity/blob/main/AutoReg_Agentic_AI_Data_Integrity.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
from sklearn.ensemble import IsolationForest

In [2]:
from google.colab import files
uploaded = files.upload()

Saving adult.csv to adult.csv


In [3]:
file = list(uploaded.keys())[0]

df = pd.read_csv(file)
df.columns = df.columns.str.strip()

df.head()

,age,workclass,fnlwgt,education,education.num,marital.status,occupation,relationship,race,sex,capital.gain,capital.loss,hours.per.week,native.country,income
0,90,?,77053,HS-grad,9,Widowed,?,Not-in-family,White,Female,0,4356,40,United-States,<=50K
1,82,Private,132870,HS-grad,9,Widowed,Exec-managerial,Not-in-family,White,Female,0,4356,18,United-States,<=50K
2,66,?,186061,Some-college,10,Widowed,?,Unmarried,Black,Female,0,4356,40,United-States,<=50K
3,54,Private,140359,7th-8th,4,Divorced,Machine-op-inspct,Unmarried,White,Female,0,3900,40,United-States,<=50K
4,41,Private,264663,Some-college,10,Separated,Prof-specialty,Own-child,White,Female,0,3900,40,United-States,<=50K


In [4]:
num_cols = df.select_dtypes(include=["int64","float64"]).columns

if len(num_cols) == 0:
    print("No numerical columns found.")

In [5]:
X = df[num_cols]

model = IsolationForest(contamination=0.05, random_state=42)
model.fit(X)

IsolationForest(contamination=0.05, random_state=42)

In [6]:
pred = model.predict(X)
df["anomaly"] = pred

anomalies = df[df["anomaly"] == -1].copy()

In [7]:
print("=== Detected Anomalies (Table) ===\n")
anomalies

=== Detected Anomalies (Table) ===



,age,workclass,fnlwgt,education,education.num,marital.status,occupation,relationship,race,sex,capital.gain,capital.loss,hours.per.week,native.country,income,anomaly
0,90,?,77053,HS-grad,9,Widowed,?,Not-in-family,White,Female,0,4356,40,United-States,<=50K,-1
1,82,Private,132870,HS-grad,9,Widowed,Exec-managerial,Not-in-family,White,Female,0,4356,18,United-States,<=50K,-1
2,66,?,186061,Some-college,10,Widowed,?,Unmarried,Black,Female,0,4356,40,United-States,<=50K,-1
3,54,Private,140359,7th-8th,4,Divorced,Machine-op-inspct,Unmarried,White,Female,0,3900,40,United-States,<=50K,-1
4,41,Private,264663,Some-college,10,Separated,Prof-specialty,Own-child,White,Female,0,3900,40,United-States,<=50K,-1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
32476,58,Self-emp-inc,181974,Doctorate,16,Never-married,Prof-specialty,Not-in-family,White,Female,0,0,99,?,<=50K,-1
32500,82,?,403910,HS-grad,9,Never-married,?,Not-in-family,White,Male,0,0,3,United-States,<=50K,-1
32528,81,?,120478,Assoc-voc,11,Divorced,?,Unmarried,White,Female,0,0,1,?,<=50K,-1
32534,30,?,33811,Bachelors,13,Never-married,?,Not-in-family,Asian-Pac-Islander,Female,0,0,99,United-States,<=50K,-1


In [8]:
summary_rows = []

for idx, row in anomalies.iterrows():
    for col in num_cols:
        value = row[col]
        mean = df[col].mean()
        std = df[col].std()

        if abs(value - mean) > 3 * std:
            summary_rows.append({
                "Row": idx,
                "Column": col,
                "Value": value,
                "Mean": round(mean,2),
                "StdDev": round(std,2)
            })

summary = pd.DataFrame(summary_rows)

In [12]:
from IPython.display import display

print("=== Anomaly Explanation Table ===\n")

if len(summary) > 0:
    display(summary)
else:
    print("No abnormal columns detected.")

=== Anomaly Explanation Table ===



,Row,Column,Value,Mean,StdDev
0,0,age,90,38.58,13.64
1,0,capital.loss,4356,87.30,402.96
2,1,age,82,38.58,13.64
3,1,capital.loss,4356,87.30,402.96
4,2,capital.loss,4356,87.30,402.96
...,...,...,...,...,...
1424,32500,age,82,38.58,13.64
1425,32500,hours.per.week,3,40.44,12.35
1426,32528,age,81,38.58,13.64
1427,32528,hours.per.week,1,40.44,12.35


In [13]:
print("Total anomalies detected:", len(anomalies))

Total anomalies detected: 1628
